In [1]:
import numpy as np
import pandas as pd
from sys import stderr
from numpy import zeros, arange, isscalar,diag, dot,eye, ix_, ones, r_, pi, flatnonzero as find
from scipy.sparse import csr_matrix
from numpy.linalg import solve

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
# === Initialization ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

case_name = 'pglib_opf_case14_ieee'
case_path = f'../excel_outputs/{case_name}.xlsx'
case = pd.read_excel(case_path, sheet_name=['baseMVA','bus','gen','gencost','branch'])

bus_to_idx = {bus: i+1 for i, bus in enumerate(case['bus'].bus_i.values)}
bus_idx = [bus_to_idx[bus] for bus in case['bus'].bus_i.values]
case['bus'].bus_i = case['bus'].bus_i.replace(bus_to_idx) # rename the bus for making PTDF
case['gen'].bus_i = case['gen'].bus_i.replace(bus_to_idx)
case['gencost'].bus_i = case['gencost'].bus_i.replace(bus_to_idx)
case['branch'].bus_i = case['branch'].bus_i.replace(bus_to_idx)
case['branch'].bus_j = case['branch'].bus_j.replace(bus_to_idx)
nbus = case['bus'].shape[0]
ngen = case['gen'].shape[0]
nbranch = case['branch'].shape[0]

# per unit p.u. conversion for cost coefficients
baseMVA = case['baseMVA'].values[0][0]
c2 = case['gencost'].c2.values * baseMVA**2
c1 = case['gencost'].c1.values * baseMVA
c0 = case['gencost'].c0.values

# calculate susceptance, conductance, admittance-square y_sq
# $Z = r + ix$ $Y = g + ib$ $Y = \frac{1}{Z} = \frac{r}{r^2 + x^2} - i\frac{x}{r^2 + x^2}$
# 1. Physics: Admittance Y = g + i*b
r = case['branch']['r'].values
x = case['branch']['x'].values
Z_sq = r**2 + x**2
g = r / Z_sq
b = -x / Z_sq
y_sq = 1 / Z_sq

# 2. Extract Line Charging, Taps, and Phase Shifts
bc = case['branch']['b'].values # MATPOWER branch 'b' is total line charging susceptance
tau = np.where(case['branch']['ratio'].values == 0, 1.0, case['branch']['ratio'].values)
theta_shift = np.radians(case['branch']['angle'].values)

# 3. Data Extraction
Gs = case['bus']['Gs'].values / baseMVA
Bs = case['bus']['Bs'].values / baseMVA
Pd = case['bus'].Pd.values / baseMVA
Qd = case['bus'].Qd.values / baseMVA

# State vector dimension D = 2 * |B|
D = 2 * nbus

# Data for QCQP

## 1. Branch Flow Matrices (Equations 14-22)

In [3]:
# Initialize lists to store matrices for all branches
M_pf = []; M_qf = []; M_pt = []; M_qt = []

# Pre-calculate derived branch elements
g11 = g / (tau**2)
g12 = g * np.cos(theta_shift) / tau
g21 = g * np.sin(theta_shift) / tau
g22 = g

b11 = (b + bc/2) / (tau**2)
b12 = b * np.cos(theta_shift) / tau
b21 = b * np.sin(theta_shift) / tau
b22 = b + bc/2

for l in range(nbranch):
    # Python uses 0-based indexing; your dictionary offset to 1-based, so we subtract 1
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    
    # Real and Imaginary indices
    i_B = i + nbus
    j_B = j + nbus

    # --- FROM END ---
    A_pf = np.zeros((D, D))
    A_pf[i, i] = g11[l]
    A_pf[i_B, i_B] = g11[l]
    A_pf[i, j] = -(g12[l] - b21[l])
    A_pf[i_B, j_B] = -(g12[l] - b21[l])
    A_pf[i, j_B] = (g21[l] + b12[l])
    A_pf[i_B, j] = -(g21[l] + b12[l])
    M_pf.append(0.5 * (A_pf + A_pf.T))

    A_qf = np.zeros((D, D))
    A_qf[i, i] = -b11[l]
    A_qf[i_B, i_B] = -b11[l]
    A_qf[i, j] = (b12[l] + g21[l])
    A_qf[i_B, j_B] = (b12[l] + g21[l])
    A_qf[i, j_B] = -(b21[l] - g12[l])
    A_qf[i_B, j] = (b21[l] - g12[l])
    M_qf.append(0.5 * (A_qf + A_qf.T))

    # --- TO END ---
    A_pt = np.zeros((D, D))
    A_pt[j, j] = g22[l]
    A_pt[j_B, j_B] = g22[l]
    A_pt[j, i] = -(g12[l] + b21[l])
    A_pt[j_B, i_B] = -(g12[l] + b21[l])
    A_pt[j, i_B] = -(g21[l] - b12[l])
    A_pt[j_B, i] = (g21[l] - b12[l])
    M_pt.append(0.5 * (A_pt + A_pt.T))

    A_qt = np.zeros((D, D))
    A_qt[j, j] = -b22[l]
    A_qt[j_B, j_B] = -b22[l]
    A_qt[j, i] = (b12[l] - g21[l])
    A_qt[j_B, i_B] = (b12[l] - g21[l])
    A_qt[j, i_B] = (b21[l] + g12[l])
    A_qt[j_B, i] = -(b21[l] + g12[l])
    M_qt.append(0.5 * (A_qt + A_qt.T))
    

## 2. Nodal Power Injection Matrices (Equations 23-28)

In [4]:
# 1. Build Standard Y_bus matrices (G_bus and B_bus)
G_bus = np.zeros((nbus, nbus))
B_bus = np.zeros((nbus, nbus))

# Add shunts to the diagonals
np.fill_diagonal(G_bus, Gs)
np.fill_diagonal(B_bus, Bs)

# Add branch contributions
for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1

    # Diagonal elements
    G_bus[i, i] += g11[l]
    G_bus[j, j] += g22[l]
    B_bus[i, i] += b11[l]
    B_bus[j, j] += b22[l]

    # Off-diagonal elements (symmetric for the line itself, but tap ratios make Y_bus asymmetric)
    G_bus[i, j] += -g12[l]
    G_bus[j, i] += -g12[l] # Assuming simplified model where g12 roughly handles the tap phase
    B_bus[i, j] += -b12[l]
    B_bus[j, i] += -b12[l]

# 2. Build the QCQP Nodal Matrices
M_p = []; M_q = []

for i in range(nbus):
    A_pi = np.zeros((D, D))
    A_qi = np.zeros((D, D))
    
    # Active Power Row mappings (Eq 25)
    A_pi[i, :nbus] = G_bus[i, :]
    A_pi[i, nbus:] = -B_bus[i, :]
    A_pi[i + nbus, :nbus] = B_bus[i, :]
    A_pi[i + nbus, nbus:] = G_bus[i, :]
    M_p.append(0.5 * (A_pi + A_pi.T))
    
    # Reactive Power Row mappings (Eq 27)
    A_qi[i, :nbus] = -B_bus[i, :]
    A_qi[i, nbus:] = -G_bus[i, :]
    A_qi[i + nbus, :nbus] = G_bus[i, :]
    A_qi[i + nbus, nbus:] = -B_bus[i, :]
    M_q.append(0.5 * (A_qi + A_qi.T))

## 3. Branch Angle Difference & Voltage Matrices (Equations 29-34)

In [5]:
M_c = []; M_s = []

for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    i_B = i + nbus
    j_B = j + nbus

    # Angle Cosine Extraction (Eq 29 & 30)
    A_c = np.zeros((D, D))
    A_c[i, j] = 1
    A_c[i_B, j_B] = 1
    M_c.append(0.5 * (A_c + A_c.T))

    # Angle Sine Extraction (Eq 31 & 32)
    A_s = np.zeros((D, D))
    A_s[i_B, j] = 1
    A_s[i, j_B] = -1
    M_s.append(0.5 * (A_s + A_s.T))

M_V = []
for i in range(nbus):
    # Voltage Magnitude Extraction (Eq 33 & 34)
    A_V = np.zeros((D, D))
    A_V[i, i] = 1
    A_V[i + nbus, i + nbus] = 1
    M_V.append(A_V) # Already symmetric

## 4. Reference Angle Vector

In [6]:
# Identify the slack bus (MATPOWER sets bus type to 3 for slack)
slack_bus_idx = case['bus'][case['bus']['type'] == 3].index[0]

a_ref = np.zeros(D)
# Force the imaginary voltage component of the slack bus to 0
a_ref[slack_bus_idx + nbus] = 1

## Tensor prep

In [7]:
# ------------------------------------------------------------
# 1) Dimensions
# ------------------------------------------------------------
# These match the sizes from your MATPOWER data
nbus = nbus
ngen = ngen
nbranch = nbranch
d = D

# ------------------------------------------------------------
# 2) Stack quadratic matrices (For ALL buses and branches)
# ------------------------------------------------------------
# Nodal power and voltage matrices [nbus, d, d]
M_p_stack = torch.stack([torch.as_tensor(M_p[i], dtype=dtype, device=device) for i in range(nbus)])
M_q_stack = torch.stack([torch.as_tensor(M_q[i], dtype=dtype, device=device) for i in range(nbus)])
M_v_stack = torch.stack([torch.as_tensor(M_V[i], dtype=dtype, device=device) for i in range(nbus)])

# Branch flow matrices [nbranch, d, d]
M_pf_stack = torch.stack([torch.as_tensor(M_pf[l], dtype=dtype, device=device) for l in range(nbranch)])
M_qf_stack = torch.stack([torch.as_tensor(M_qf[l], dtype=dtype, device=device) for l in range(nbranch)])
M_pt_stack = torch.stack([torch.as_tensor(M_pt[l], dtype=dtype, device=device) for l in range(nbranch)])
M_qt_stack = torch.stack([torch.as_tensor(M_qt[l], dtype=dtype, device=device) for l in range(nbranch)])

# Angle difference matrices [nbranch, d, d]
M_c_stack = torch.stack([torch.as_tensor(M_c[l], dtype=dtype, device=device) for l in range(nbranch)])
M_s_stack = torch.stack([torch.as_tensor(M_s[l], dtype=dtype, device=device) for l in range(nbranch)])

# ------------------------------------------------------------
# 3) Symmetrize matrices (Required for stable Autograd)
# ------------------------------------------------------------
M_p_stack = 0.5 * (M_p_stack + M_p_stack.transpose(-1, -2))
M_q_stack = 0.5 * (M_q_stack + M_q_stack.transpose(-1, -2))
M_v_stack = 0.5 * (M_v_stack + M_v_stack.transpose(-1, -2))

M_pf_stack = 0.5 * (M_pf_stack + M_pf_stack.transpose(-1, -2))
M_qf_stack = 0.5 * (M_qf_stack + M_qf_stack.transpose(-1, -2))
M_pt_stack = 0.5 * (M_pt_stack + M_pt_stack.transpose(-1, -2))
M_qt_stack = 0.5 * (M_qt_stack + M_qt_stack.transpose(-1, -2))
M_c_stack = 0.5 * (M_c_stack + M_c_stack.transpose(-1, -2))
M_s_stack = 0.5 * (M_s_stack + M_s_stack.transpose(-1, -2))

# ------------------------------------------------------------
# 4) The C_g Matrix (Mapping Generators to Buses)
# ------------------------------------------------------------
# Shape: [nbus, ngen]
C_g = torch.zeros((nbus, ngen), dtype=dtype, device=device)
for gen_idx, bus_i in enumerate(case['gen']['bus_i'].values):
    bus_idx = int(bus_i) - 1 # convert to 0-based index
    C_g[bus_idx, gen_idx] = 1.0

# ------------------------------------------------------------
# 5) Vectors: Demands, Limits, and Reference
# ------------------------------------------------------------
Pd_bus = np.asarray(case['bus'].Pd.values, dtype=np.float32) / baseMVA
Qd_bus = np.asarray(case['bus'].Qd.values, dtype=np.float32) / baseMVA

pmax = np.asarray(case['gen'].Pmax.values, dtype=np.float32) / baseMVA
pmin = np.asarray(case['gen'].Pmin.values, dtype=np.float32) / baseMVA
qmax = np.asarray(case['gen'].Qmax.values, dtype=np.float32) / baseMVA
qmin = np.asarray(case['gen'].Qmin.values, dtype=np.float32) / baseMVA

# Apparent power branch limits (s_max)
smax = np.asarray(case['branch'].rateA.values, dtype=np.float32) / baseMVA
smax[smax == 0] = 9999.0  # Handle unconstrained lines gracefully

# Branch angle limits (converted to radians)
angmax = np.radians(np.asarray(case['branch'].angmax.values, dtype=np.float32))
angmin = np.radians(np.asarray(case['branch'].angmin.values, dtype=np.float32))

Vmax_arr = np.asarray(case['bus'].Vmax.values, dtype=np.float32)
Vmin_arr = np.asarray(case['bus'].Vmin.values, dtype=np.float32)

# ------------------------------------------------------------
# 6) Final problem dictionary for the PINN loss
# ------------------------------------------------------------
problem = {
    # Quadratic Matrices
    "M_p": M_p_stack,
    "M_q": M_q_stack,
    "M_v": M_v_stack,
    "M_pf": M_pf_stack,
    "M_qf": M_qf_stack,
    "M_pt": M_pt_stack,
    "M_qt": M_qt_stack,
    "M_c": M_c_stack,
    "M_s": M_s_stack,

    # Incidence Matrix
    "C_g": C_g,

    # Base Vectors
    "Pd": torch.as_tensor(Pd_bus, dtype=dtype, device=device),
    "Qd": torch.as_tensor(Qd_bus, dtype=dtype, device=device),
    
    "pmax": torch.as_tensor(pmax, dtype=dtype, device=device),
    "pmin": torch.as_tensor(pmin, dtype=dtype, device=device),
    "qmax": torch.as_tensor(qmax, dtype=dtype, device=device),
    "qmin": torch.as_tensor(qmin, dtype=dtype, device=device),
    
    "smax": torch.as_tensor(smax, dtype=dtype, device=device),
    "angmax": torch.as_tensor(angmax, dtype=dtype, device=device),
    "angmin": torch.as_tensor(angmin, dtype=dtype, device=device),
    
    "Vmax": torch.as_tensor(Vmax_arr, dtype=dtype, device=device),
    "Vmin": torch.as_tensor(Vmin_arr, dtype=dtype, device=device),
    
    # Add the cost coefficients
    "c2": torch.tensor(c2, dtype=dtype, device=device),
    "c1": torch.tensor(c1, dtype=dtype, device=device),
    "c0": torch.tensor(c0, dtype=dtype, device=device),
        
    # Anchor vector (Ensure a_ref from our earlier discussion is defined)
    "a_ref": torch.as_tensor(a_ref, dtype=dtype, device=device),

    # Metadata
    "nbus": nbus,
    "ngen": ngen,
    "nbranch": nbranch
}

print("Constructed PINN problem data for QCQP:")
print(f"  nbus    = {nbus}")
print(f"  ngen    = {ngen}")
print(f"  nbranch = {nbranch}")
print(f"  M_pf, M_qf shape = {tuple(problem['M_pf'].shape)}, {tuple(problem['M_qf'].shape)}")
print(f"  M_pt, M_qt shape = {tuple(problem['M_pt'].shape)}, {tuple(problem['M_qt'].shape)}")
print(f"  C_g shape  = {tuple(problem['C_g'].shape)}")
print(f"  M_p, M_q shape  = {tuple(problem['M_p'].shape)}, {tuple(problem['M_q'].shape)}")
print(f"  M_s, M_c shape  = {tuple(problem['M_s'].shape)}, {tuple(problem['M_c'].shape)}")
print(f"  M_V shape  = {tuple(problem['M_v'].shape)}")

Constructed PINN problem data for QCQP:
  nbus    = 14
  ngen    = 5
  nbranch = 20
  M_pf, M_qf shape = (20, 28, 28), (20, 28, 28)
  M_pt, M_qt shape = (20, 28, 28), (20, 28, 28)
  C_g shape  = (14, 5)
  M_p, M_q shape  = (14, 28, 28), (14, 28, 28)
  M_s, M_c shape  = (20, 28, 28), (20, 28, 28)
  M_V shape  = (14, 28, 28)


# Random demand

In [8]:
def gaussian_batch(base_tensor, batch_size, variation_std=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with Gaussian random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation_std (float): standard deviation of Gaussian noise, relative to base_tensor
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    # Expand base tensor to batch
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Gaussian noise scaled by base_tensor
    noise = variation_std * base_tensor.unsqueeze(0) * torch.randn_like(base_batch)
    
    # Perturbed batch
    batch = base_batch + noise
    
    # Clamp if necessary
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch

def uniform_batch(base_tensor, batch_size, variation=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with uniform random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation (float): maximum relative deviation (±variation)
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Uniform noise in [-variation, +variation]
    noise = (2 * torch.rand_like(base_batch) - 1) * variation
    
    batch = base_batch * (1 + noise)
    
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch


# Base line model

## 1. PINN Architecture

In [9]:
class baselineQCQPMLP(nn.Module):
    """
    Input:
        Pd: [B, nbus]
        Qd: [B, nbus]
    Output:
        v:  [B, 2*nbus] (Rectangular voltages)
        pg: [B, ngen]   (Active generation)
        qg: [B, ngen]   (Reactive generation)
    """
    def __init__(self, nbus: int, ngen: int, slack_imag_idx: int, hidden: int = 512):
        super().__init__()
        self.nbus = nbus
        self.ngen = ngen
        self.in_dim = 2 * nbus
        self.out_dim_v = 2 * nbus
        self.out_dim_g = 2 * ngen 
        self.slack_imag_idx = int(slack_imag_idx)

        # Core MLP
        self.net = nn.Sequential(
            nn.Linear(self.in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, self.out_dim_v + self.out_dim_g),
        )

    def forward(self, Pd: torch.Tensor, Qd: torch.Tensor, problem: dict) -> tuple:
        B = Pd.shape[0]
        x = torch.cat([Pd, Qd], dim=-1)
        raw = self.net(x)

        # 1. Slice outputs
        v_raw = raw[:, :self.out_dim_v]
        g_raw = raw[:, self.out_dim_v:]
        
        pg_raw = g_raw[:, :self.ngen]
        qg_raw = g_raw[:, self.ngen:]

        # 2. Bound Voltages to [-Vmax, Vmax] using Tanh for smooth gradients
        Vmax_b = problem["Vmax"].reshape(1, -1).expand(B, -1)
        Vmax_full = torch.cat([Vmax_b, Vmax_b], dim=-1) # For real and imaginary parts
        v = torch.tanh(v_raw) * Vmax_full

        # Constraint (2m): Enforce slack imaginary part = 0 exactly
        v_clone = v.clone()
        v_clone[:, self.slack_imag_idx] = 0.0
        v = v_clone

        # 3. Bound Generation strictly between [min, max] using Sigmoid
        pmax_b = problem["pmax"].reshape(1, -1).expand(B, -1)
        pmin_b = problem["pmin"].reshape(1, -1).expand(B, -1)
        qmax_b = problem["qmax"].reshape(1, -1).expand(B, -1)
        qmin_b = problem["qmin"].reshape(1, -1).expand(B, -1)

        pg = pmin_b + torch.sigmoid(pg_raw) * (pmax_b - pmin_b)
        qg = qmin_b + torch.sigmoid(qg_raw) * (qmax_b - qmin_b)

        return v, pg, qg

## 2. The QCQP Loss Function

In [10]:
def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [K, d, d] -> [B, K]
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_qcqp_loss(model: nn.Module, Pd_batch: torch.Tensor, Qd_batch: torch.Tensor, problem: dict, weights: dict):
    B = Pd_batch.shape[0]
    
    # Predict continuous variables (bounded by construction)
    v, pg, qg = model(Pd_batch, Qd_batch, problem)

    # --------------------------------------------------------
    # Unpack Problem Matrices
    # --------------------------------------------------------
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"] # [nbus, ngen]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    
    # Expand Generator Limits for batch
    pmax = problem["pmax"].unsqueeze(0).expand(B, -1)
    pmin = problem["pmin"].unsqueeze(0).expand(B, -1)
    qmax = problem["qmax"].unsqueeze(0).expand(B, -1)
    qmin = problem["qmin"].unsqueeze(0).expand(B, -1)

    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1)

    # --------------------------------------------------------
    # Evaluate Quadratic Forms
    # --------------------------------------------------------
    vp = quad_batch_stack(v, M_p)   # [B, nbus]
    vq = quad_batch_stack(v, M_q)
    
    pf = quad_batch_stack(v, M_pf)  # 2b
    qf = quad_batch_stack(v, M_qf)  # 2c
    pt = quad_batch_stack(v, M_pt)  # 2d
    qt = quad_batch_stack(v, M_qt)  # 2e
    
    vc = quad_batch_stack(v, M_c)
    vs = quad_batch_stack(v, M_s)
    vv = quad_batch_stack(v, M_v)

    # --------------------------------------------------------
    # Constraints Mapping (Equations 2h - 2m)
    # --------------------------------------------------------
    
    # 1. Objective (Eq 2a) - Minimize total active generation
    cost_per_gen = c2 * (pg ** 2) + c1 * pg + c0
    obj = cost_per_gen.sum(dim=1).mean()
    
    # 2. Branch Thermal Limits (Eq 2f & 2g): p^2 + q^2 <= smax^2
    g_sf = (pf**2 + qf**2) - smax**2
    g_st = (pt**2 + qt**2) - smax**2
    
    # 3. Nodal Power Balance (Eq 2h & 2i): [C_g * pg]_i - pd_i = v^T M_p v
    # pg @ C_g.T automatically maps the ngen generations to their nbus locations
    h_p = (pg @ C_g.T) - Pd_batch - vp
    h_q = (qg @ C_g.T) - Qd_batch - vq
    # Generator Limits
    g_pg_max = pg - pmax
    g_pg_min = pmin - pg
    g_qg_max = qg - qmax
    g_qg_min = qmin - qg

    # 4. Angle Difference Stability (Eq 2j & 2k)
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc

    # 5. Voltage Magnitude Security (Eq 2l): Vmin^2 <= v^T M_v v <= Vmax^2
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv

    # --------------------------------------------------------
    # Compute Penalties
    # --------------------------------------------------------
    loss_eq_p = h_p.pow(2).mean()
    loss_eq_q = h_q.pow(2).mean()
    
    loss_thermal = F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean()
    loss_ang = F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean()
    loss_v = F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean()

    total_loss = (
        weights["eq_p"] * loss_eq_p +
        weights["eq_q"] * loss_eq_q +
        weights["thermal"] * loss_thermal +
        weights["ang"] * loss_ang +
        weights["v"] * loss_v +
        weights["obj"] * obj
    )

    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "obj_cost": obj.detach().item(),
        
        "max_h_p": h_p.abs().max().detach().item(),
        "max_h_q": h_q.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf).max(), F.relu(g_st).max()).detach().item(),
        "max_v_viol": torch.max(F.relu(g_v_max).max(), F.relu(g_v_min).max()).detach().item(),
        
        # This will consistently report 0.0 in the baseline!
        "max_gen_viol": torch.max(
            torch.max(F.relu(g_pg_max).max(), F.relu(g_pg_min).max()),
            torch.max(F.relu(g_qg_max).max(), F.relu(g_qg_min).max())
        ).detach().item()
    }
    return total_loss, diagnostics

## 3. Training

In [14]:
# Extract the exact index where a_ref is 1 (the slack bus imaginary component)
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model = baselineQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Define penalty weights (tune these based on convergence)
loss_weights = {
    "eq_p": 1000.0,
    "eq_q": 1000.0,
    "thermal": 1.0,
    "ang": 1.0,
    "v": 1.0,
    "obj": 0.01
}

# Example Training Loop
for epoch in range(1000):
    # Base demands from problem dict
    Pd_base = problem["Pd"]
    Qd_base = problem["Qd"]
    
    # Generate random scenario batch
    Pd_batch = gaussian_batch(Pd_base, batch_size=32)
    Qd_batch = gaussian_batch(Qd_base, batch_size=32)
    
    optimizer.zero_grad()
    loss, diag = compute_qcqp_loss(model, Pd_batch, Qd_batch, problem, loss_weights)
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f}")

Epoch    0 | Cost: 2005.20 | Max P-Miss: 1.6850 | Max Q-Miss: 0.2961 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  100 | Cost: 1060.27 | Max P-Miss: 0.3046 | Max Q-Miss: 0.0815 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  200 | Cost: 1065.33 | Max P-Miss: 0.2640 | Max Q-Miss: 0.1169 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  300 | Cost: 1168.74 | Max P-Miss: 0.2229 | Max Q-Miss: 0.1136 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  400 | Cost: 1058.77 | Max P-Miss: 0.3051 | Max Q-Miss: 0.2230 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  500 | Cost: 1211.48 | Max P-Miss: 0.2844 | Max Q-Miss: 0.0849 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  600 | Cost: 1275.21 | Max P-Miss: 0.1880 | Max Q-Miss: 0.1092 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  700 | Cost: 1307.38 | Max P-Miss: 0.1960 | Max Q-Miss: 0.0852 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch  800 | Cost: 1305.17 | Max P-Miss: 0.1980 | Max Q-Miss: 0.1179 | Max Gen Viol: 0.0

# PINN paper

## Rahul's Branched Architecture in PyTorch

In [ ]:
class RahulSinglePINN_Smax(nn.Module):
    """
    Version 2: Single Neural Network for all variables (Primal + Dual).
    """
    def __init__(self, nbus, ngen, nbranch, hidden_dim=512):
        super().__init__()
        self.nbus = nbus
        self.ngen = ngen
        self.nbranch = nbranch
        
        in_dim = 2 * nbus # Pd and Qd
        
        # Calculate total output dimension
        self.dim_v = 2 * nbus
        self.dim_g = 2 * ngen # pg and qg
        self.num_duals = (4 * nbus) + (4 * nbranch) + (4 * ngen)
        
        out_dim = self.dim_v + self.dim_g + self.num_duals
        
        # A SINGLE Neural Network for everything
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, Pd, Qd):
        x = torch.cat([Pd, Qd], dim=-1)
        
        # Single forward pass
        raw = self.net(x)
        
        # ----------------------------------------------------
        # 1. Slice Primal Variables (Unbounded)
        # ----------------------------------------------------
        idx = 0
        v = raw[:, idx : idx + self.dim_v]; idx += self.dim_v
        
        pq = raw[:, idx : idx + self.dim_g]; idx += self.dim_g
        pg = pq[:, :self.ngen]
        qg = pq[:, self.ngen:]
        
        # ----------------------------------------------------
        # 2. Slice Dual Variables (Lagrange Multipliers)
        # ----------------------------------------------------
        lam_p = raw[:, idx : idx+self.nbus]; idx += self.nbus
        lam_q = raw[:, idx : idx+self.nbus]; idx += self.nbus
        
        mu_sf = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        mu_st = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        
        mu_ang_max = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        mu_ang_min = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        
        mu_v_max = raw[:, idx : idx+self.nbus]; idx += self.nbus
        mu_v_min = raw[:, idx : idx+self.nbus]; idx += self.nbus
        
        mu_pg_max = raw[:, idx : idx+self.ngen]; idx += self.ngen
        mu_pg_min = raw[:, idx : idx+self.ngen]; idx += self.ngen
        mu_qg_max = raw[:, idx : idx+self.ngen]; idx += self.ngen
        mu_qg_min = raw[:, idx : idx+self.ngen]; idx += self.ngen
        
        return (v, pg, qg, lam_p, lam_q, mu_sf, mu_st, 
                mu_ang_max, mu_ang_min, mu_v_max, mu_v_min, 
                mu_pg_max, mu_pg_min, mu_qg_max, mu_qg_min)

## Rahul's Primal-Dual KKT Loss Function

In [13]:
# Helper function to compute (M * v) for batches
def batch_Mv(M: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    # M: [K, d, d], v: [B, d] -> [B, K, d]
    return torch.einsum('kij,bj->bki', M, v)

def compute_rahul_kkt_smax_loss(model, Pd_batch, Qd_batch, problem, weights):
    B = Pd_batch.shape[0]
    
    # Forward Pass
    (v, pg, qg, lam_p, lam_q, mu_sf, mu_st, 
     mu_ang_max, mu_ang_min, mu_v_max, mu_v_min, 
     mu_pg_max, mu_pg_min, mu_qg_max, mu_qg_min) = model(Pd_batch, Qd_batch)

    # Matrices & Limits
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    pmax = problem["pmax"].unsqueeze(0).expand(B, -1)
    pmin = problem["pmin"].unsqueeze(0).expand(B, -1)
    qmax = problem["qmax"].unsqueeze(0).expand(B, -1)
    qmin = problem["qmin"].unsqueeze(0).expand(B, -1)
    
    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1)

    # --------------------------------------------------------
    # A. PRIMAL EVALUATIONS
    # --------------------------------------------------------
    vp = quad_batch_stack(v, M_p); vq = quad_batch_stack(v, M_q)
    pf = quad_batch_stack(v, M_pf); qf = quad_batch_stack(v, M_qf)
    pt = quad_batch_stack(v, M_pt); qt = quad_batch_stack(v, M_qt)
    vc = quad_batch_stack(v, M_c); vs = quad_batch_stack(v, M_s)
    vv = quad_batch_stack(v, M_v)

    # Equations
    h_p = (pg @ C_g.T) - Pd_batch - vp
    h_q = (qg @ C_g.T) - Qd_batch - vq
    g_sf = (pf**2 + qf**2) - smax**2
    g_st = (pt**2 + qt**2) - smax**2
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv
    g_pg_max = pg - pmax; g_pg_min = pmin - pg
    g_qg_max = qg - qmax; g_qg_min = qmin - qg

    # --------------------------------------------------------
    # CALCULATE OBJECTIVE COST
    # --------------------------------------------------------
    cost_per_gen = c2 * (pg ** 2) + c1 * pg + c0
    obj = cost_per_gen.sum(dim=1).mean()

    # --------------------------------------------------------
    # B. COMPLEMENTARY SLACKNESS (mu * g == 0)
    # --------------------------------------------------------
    cs_loss = (
        (mu_sf * g_sf).pow(2).mean() + (mu_st * g_st).pow(2).mean() +
        (mu_ang_max * g_ang_max).pow(2).mean() + (mu_ang_min * g_ang_min).pow(2).mean() +
        (mu_v_max * g_v_max).pow(2).mean() + (mu_v_min * g_v_min).pow(2).mean() +
        (mu_pg_max * g_pg_max).pow(2).mean() + (mu_pg_min * g_pg_min).pow(2).mean() +
        (mu_qg_max * g_qg_max).pow(2).mean() + (mu_qg_min * g_qg_min).pow(2).mean()
    )

    # --------------------------------------------------------
    # C. DUAL FEASIBILITY (mu >= 0)
    # --------------------------------------------------------
    dual_feas_loss = (
        F.relu(-mu_sf).pow(2).mean() + F.relu(-mu_st).pow(2).mean() +
        F.relu(-mu_ang_max).pow(2).mean() + F.relu(-mu_ang_min).pow(2).mean() +
        F.relu(-mu_v_max).pow(2).mean() + F.relu(-mu_v_min).pow(2).mean() +
        F.relu(-mu_pg_max).pow(2).mean() + F.relu(-mu_pg_min).pow(2).mean() +
        F.relu(-mu_qg_max).pow(2).mean() + F.relu(-mu_qg_min).pow(2).mean()
    )

    # --------------------------------------------------------
    # D. KKT STATIONARITY (Analytical Gradients = 0)
    # --------------------------------------------------------
    dL_dpg = (2 * c2 * pg) + c1 + (lam_p @ C_g) + mu_pg_max - mu_pg_min
    dL_dqg = (lam_q @ C_g) + mu_qg_max - mu_qg_min

    # Exact Analytical Gradient w.r.t voltage (v)
    dL_dv_p = -2 * torch.einsum('bk,bki->bi', lam_p, batch_Mv(M_p, v))
    dL_dv_q = -2 * torch.einsum('bk,bki->bi', lam_q, batch_Mv(M_q, v))
    
    # The Quartic Analytical Gradient for smax: 4 * mu * (p * Mp*v + q * Mq*v)
    dL_dv_sf = 4 * torch.einsum('bk,bk,bki->bi', mu_sf, pf, batch_Mv(M_pf, v)) + \
               4 * torch.einsum('bk,bk,bki->bi', mu_sf, qf, batch_Mv(M_qf, v))
    dL_dv_st = 4 * torch.einsum('bk,bk,bki->bi', mu_st, pt, batch_Mv(M_pt, v)) + \
               4 * torch.einsum('bk,bk,bki->bi', mu_st, qt, batch_Mv(M_qt, v))
    
    dL_dv_vmax = 2 * torch.einsum('bk,bki->bi', mu_v_max, batch_Mv(M_v, v))
    dL_dv_vmin = -2 * torch.einsum('bk,bki->bi', mu_v_min, batch_Mv(M_v, v))
    
    M_s_v = batch_Mv(M_s, v); M_c_v = batch_Mv(M_c, v)
    t_max = torch.tan(angmax).unsqueeze(-1); t_min = torch.tan(angmin).unsqueeze(-1)
    
    dL_dv_angmax = torch.einsum('bk,bki->bi', mu_ang_max, 2 * M_s_v - 2 * t_max * M_c_v)
    dL_dv_angmin = torch.einsum('bk,bki->bi', mu_ang_min, 2 * t_min * M_c_v - 2 * M_s_v)

    dL_dv = dL_dv_p + dL_dv_q + dL_dv_sf + dL_dv_st + dL_dv_vmax + dL_dv_vmin + dL_dv_angmax + dL_dv_angmin
    
    stationarity_loss = dL_dpg.pow(2).mean() + dL_dqg.pow(2).mean() + dL_dv.pow(2).mean()

    # --------------------------------------------------------
    # E. PRIMAL LOSS (Actual Physical Violations)
    # --------------------------------------------------------
    primal_loss = (
        h_p.pow(2).mean() + h_q.pow(2).mean() +
        F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean() +
        F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean() +
        F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean() +
        F.relu(g_pg_max).pow(2).mean() + F.relu(g_pg_min).pow(2).mean() +
        F.relu(g_qg_max).pow(2).mean() + F.relu(g_qg_min).pow(2).mean()
    )

    total_loss = (
        weights["primal"] * primal_loss +
        weights["cs"] * cs_loss +
        weights["dual_feas"] * dual_feas_loss +
        weights["stationarity"] * stationarity_loss
    )

    # --------------------------------------------------------
    # DIAGNOSTICS FOR BENCHMARKING
    # --------------------------------------------------------
    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_primal": primal_loss.detach().item(),
        "loss_kkt_stat": stationarity_loss.detach().item(),
        "loss_kkt_cs": cs_loss.detach().item(),
        
        "obj_cost": obj.detach().item(),
        "max_h_p": h_p.abs().max().detach().item(),
        "max_h_q": h_q.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf).max(), F.relu(g_st).max()).detach().item(),
        "max_gen_viol": torch.max(
            torch.max(F.relu(g_pg_max).max(), F.relu(g_pg_min).max()),
            torch.max(F.relu(g_qg_max).max(), F.relu(g_qg_min).max())
        ).detach().item()
    }

    return total_loss, diagnostics

## Training

In [14]:
model_rahul = RahulSinglePINN_Smax(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    nbranch=problem["nbranch"]
).to(device)

optimizer_rahul = optim.Adam(model_rahul.parameters(), lr=1e-3)

# Note: Stationarity loss can explode easily because of the quartic s_max gradient.
# You may need to tune "stationarity" weight down if the loss goes to NaN.
loss_weights_rahul = {
    "primal": 10.0,         
    "cs": 1.0,              
    "dual_feas": 1.0,       
    "stationarity": 0.01     
}

print("Starting Training: Rahul's KKT PINN (Smax)")

for epoch in range(1000):
    Pd_batch = torch.clamp(gaussian_batch(problem["Pd"], batch_size=32), min=0.0)
    Qd_batch = torch.clamp(gaussian_batch(problem["Qd"], batch_size=32), min=0.0)
    
    optimizer_rahul.zero_grad()
    
    loss, diag = compute_rahul_kkt_smax_loss(
        model=model_rahul, 
        Pd_batch=Pd_batch, 
        Qd_batch=Qd_batch, 
        problem=problem, 
        weights=loss_weights_rahul
    )
    
    loss.backward()
    
    # Critical: Clip gradients to prevent the quartic s_max derivatives from exploding
    torch.nn.utils.clip_grad_norm_(model_rahul.parameters(), 10.0)
    optimizer_rahul.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f}")

Starting Training: Rahul's KKT PINN (Smax)
Epoch    0 | Cost: -115.79 | Max P-Miss: 1.1075 | Max Q-Miss: 0.3050 | Max Gen Viol: 0.0642 | Max Thermal: 0.0000
Epoch  100 | Cost:  489.19 | Max P-Miss: 0.9449 | Max Q-Miss: 0.5133 | Max Gen Viol: 0.1184 | Max Thermal: 0.0000
Epoch  200 | Cost: -6842.57 | Max P-Miss: 65.9533 | Max Q-Miss: 230.1347 | Max Gen Viol: 6.2534 | Max Thermal: 18421.9941
Epoch  300 | Cost:  232.13 | Max P-Miss: 8.7274 | Max Q-Miss: 11.3294 | Max Gen Viol: 0.6093 | Max Thermal: 72.9117
Epoch  400 | Cost: 1587.20 | Max P-Miss: 4.9296 | Max Q-Miss: 11.0739 | Max Gen Viol: 1.3202 | Max Thermal: 27.2318
Epoch  500 | Cost: -378.50 | Max P-Miss: 3.9585 | Max Q-Miss: 27.6238 | Max Gen Viol: 1.2725 | Max Thermal: 193.3492
Epoch  600 | Cost:  248.92 | Max P-Miss: 1.2417 | Max Q-Miss: 2.3050 | Max Gen Viol: 0.1803 | Max Thermal: 0.0000
Epoch  700 | Cost: -370.39 | Max P-Miss: 5.9411 | Max Q-Miss: 25.6593 | Max Gen Viol: 3.2468 | Max Thermal: 183.9127
Epoch  800 | Cost:  772.70 

# DC3


## The DC3 Loss Function with Correction Loop

In [32]:
# Helper function (same as before)
def batch_Mv(M: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    return torch.einsum('kij,bj->bki', M, v)

def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_dc3_qcqp_smax_loss(model, Pd_batch, Qd_batch, problem, weights, corr_steps=10, corr_lr=1e-3):
    B = Pd_batch.shape[0]
    
    # --------------------------------------------------------
    # 1. FORWARD PASS (Network Prediction)
    # --------------------------------------------------------
    # Using your baseline MLP that naturally bounds pg and qg
    v_pred, pg_pred, qg_pred = model(Pd_batch, Qd_batch, problem)

    # Unpack Problem Matrices
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1) if "c0" in problem else 0.0

    # --------------------------------------------------------
    # 2. DC3 CORRECTION PHASE (Inner Optimization Loop)
    # --------------------------------------------------------
    # We clone the predictions and detach them from the neural network graph
    v_c = v_pred.detach().clone().requires_grad_(True)
    pg_c = pg_pred.detach().clone().requires_grad_(True)
    qg_c = qg_pred.detach().clone().requires_grad_(True)
    
    # Define an inner optimizer solely for the correction steps
    optimizer_corr = torch.optim.Adam([v_c, pg_c, qg_c], lr=corr_lr)
    
    with torch.enable_grad():
        for _ in range(corr_steps):
            optimizer_corr.zero_grad()
            
            # Evaluate Physics on the *Correction* variables
            vp_c = quad_batch_stack(v_c, M_p)
            vq_c = quad_batch_stack(v_c, M_q)
            pf_c = quad_batch_stack(v_c, M_pf); qf_c = quad_batch_stack(v_c, M_qf)
            pt_c = quad_batch_stack(v_c, M_pt); qt_c = quad_batch_stack(v_c, M_qt)
            vc_c = quad_batch_stack(v_c, M_c); vs_c = quad_batch_stack(v_c, M_s)
            vv_c = quad_batch_stack(v_c, M_v)
            
            # Constraints
            h_p_c = (pg_c @ C_g.T) - Pd_batch - vp_c
            h_q_c = (qg_c @ C_g.T) - Qd_batch - vq_c
            g_sf_c = (pf_c**2 + qf_c**2) - smax**2
            g_st_c = (pt_c**2 + qt_c**2) - smax**2
            g_ang_min_c = torch.tan(angmin) * vc_c - vs_c
            g_ang_max_c = vs_c - torch.tan(angmax) * vc_c
            g_v_max_c = vv_c - (Vmax**2)
            g_v_min_c = (Vmin**2) - vv_c
            
            # Sum up all violations to create a repair gradient
            viol_loss = (
                h_p_c.pow(2).mean() + h_q_c.pow(2).mean() +
                F.relu(g_sf_c).pow(2).mean() + F.relu(g_st_c).pow(2).mean() +
                F.relu(g_ang_min_c).pow(2).mean() + F.relu(g_ang_max_c).pow(2).mean() +
                F.relu(g_v_max_c).pow(2).mean() + F.relu(g_v_min_c).pow(2).mean()
            )
            
            viol_loss.backward()
            optimizer_corr.step()

    # --------------------------------------------------------
    # 3. STANDARD PRIMAL EVALUATION (On original NN output)
    # --------------------------------------------------------
    vp = quad_batch_stack(v_pred, M_p); vq = quad_batch_stack(v_pred, M_q)
    pf = quad_batch_stack(v_pred, M_pf); qf = quad_batch_stack(v_pred, M_qf)
    pt = quad_batch_stack(v_pred, M_pt); qt = quad_batch_stack(v_pred, M_qt)
    vc = quad_batch_stack(v_pred, M_c); vs = quad_batch_stack(v_pred, M_s)
    vv = quad_batch_stack(v_pred, M_v)

    h_p = (pg_pred @ C_g.T) - Pd_batch - vp
    h_q = (qg_pred @ C_g.T) - Qd_batch - vq
    g_sf = (pf**2 + qf**2) - smax**2
    g_st = (pt**2 + qt**2) - smax**2
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv

    cost_per_gen = c2 * (pg_pred ** 2) + c1 * pg_pred + c0
    obj = cost_per_gen.sum(dim=1).mean()

    primal_loss = (
        h_p.pow(2).mean() + h_q.pow(2).mean() +
        F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean() +
        F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean() +
        F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean()
    )

    # --------------------------------------------------------
    # 4. DC3 TARGET LOSS
    # --------------------------------------------------------
    # Penalize distance between Neural Network output and the Repaired Target
    dc3_corr_loss = (
        F.mse_loss(v_pred, v_c.detach()) + 
        F.mse_loss(pg_pred, pg_c.detach()) + 
        F.mse_loss(qg_pred, qg_c.detach())
    )

    total_loss = (weights["primal"] * primal_loss) + (weights["obj"] * obj) + (weights["dc3_corr"] * dc3_corr_loss)

    # --------------------------------------------------------
    # DIAGNOSTICS FOR BENCHMARKING
    # --------------------------------------------------------
    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_primal": primal_loss.detach().item(),
        "loss_dc3_corr": dc3_corr_loss.detach().item(),
        "obj_cost": obj.detach().item(),
        
        "max_h_p": h_p.abs().max().detach().item(),
        "max_h_q": h_q.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf).max(), F.relu(g_st).max()).detach().item(),
        "max_v_viol": torch.max(F.relu(g_v_max).max(), F.relu(g_v_min).max()).detach().item(),
        "max_gen_viol": 0.0 # Baseline model bounds generators by construction!
    }

    return total_loss, diagnostics

## The DC3 Training Loop

In [ ]:
# Use the exact same model architecture as the Baseline for a fair comparison
# (Extract slack index again if you haven't already)
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model_dc3 = baselineQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

optimizer_dc3 = optim.Adam(model_dc3.parameters(), lr=1e-3)

# DC3 relies heavily on the correction target weight
loss_weights_dc3 = {
    "primal": 10.0,       # Soft penalty on constraints
    "obj": 0.01,          # Generation cost weight
    "dc3_corr": 50.0      # Heavy weight pushing predictions towards the repaired targets
}

print("Starting Training: DC3 QCQP (With Unrolled Correction)")

for epoch in range(1000):
    Pd_batch = torch.clamp(gaussian_batch(problem["Pd"], batch_size=32), min=0.0)
    Qd_batch = torch.clamp(gaussian_batch(problem["Qd"], batch_size=32), min=0.0)
    
    optimizer_dc3.zero_grad()
    
    # Notice we pass corr_steps=5. This adds computational overhead but heavily accelerates learning feasibility!
    loss, diag = compute_dc3_qcqp_smax_loss(
        model=model_dc3, 
        Pd_batch=Pd_batch, 
        Qd_batch=Qd_batch, 
        problem=problem, 
        weights=loss_weights_dc3,
        corr_steps=5,     # Adjust between 5-20 based on RAM / Time
        corr_lr=1e-2      # Inner loop learning rate
    )
    
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model_dc3.parameters(), 10.0)
    optimizer_dc3.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f} | DC3 Corr: {diag['loss_dc3_corr']:.4f}")

Starting Training: DC3 QCQP (With Unrolled Correction)
Epoch    0 | Cost: 2014.06 | Max P-Miss: 1.6726 | Max Q-Miss: 0.1824 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000 | DC3 Corr: 0.0049
Epoch  100 | Cost:    8.98 | Max P-Miss: 0.5969 | Max Q-Miss: 0.6077 | Max Gen Viol: 0.0000 | Max Thermal: 0.0400 | DC3 Corr: 0.0047
Epoch  200 | Cost:   19.99 | Max P-Miss: 1.0676 | Max Q-Miss: 0.3755 | Max Gen Viol: 0.0000 | Max Thermal: 0.1644 | DC3 Corr: 0.0046
Epoch  300 | Cost:   13.57 | Max P-Miss: 0.6509 | Max Q-Miss: 0.6953 | Max Gen Viol: 0.0000 | Max Thermal: 0.0097 | DC3 Corr: 0.0047
Epoch  400 | Cost:   12.73 | Max P-Miss: 0.7827 | Max Q-Miss: 0.5179 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000 | DC3 Corr: 0.0045
Epoch  500 | Cost:    8.77 | Max P-Miss: 0.6409 | Max Q-Miss: 0.4670 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000 | DC3 Corr: 0.0046
Epoch  600 | Cost:    5.13 | Max P-Miss: 0.6125 | Max Q-Miss: 0.4080 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000 | DC3 Corr: 0.0046
Epoch  700 | C

# FSNet

## The FSNet QCQP Loss Function

In [38]:
# Helper functions
def batch_Mv(M: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    return torch.einsum('kij,bj->bki', M, v)

def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_fsnet_qcqp_smax_loss(model, Pd_batch, Qd_batch, problem, weights, seek_steps=5, seek_lr=1e-3):
    B = Pd_batch.shape[0]
    
    # --------------------------------------------------------
    # 1. FORWARD PASS (Initial Guess y_0)
    # --------------------------------------------------------
    v_0, pg_0, qg_0 = model(Pd_batch, Qd_batch, problem)

    # Unpack Problem Matrices
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    pmax = problem["pmax"].unsqueeze(0).expand(B, -1)
    pmin = problem["pmin"].unsqueeze(0).expand(B, -1)
    qmax = problem["qmax"].unsqueeze(0).expand(B, -1)
    qmin = problem["qmin"].unsqueeze(0).expand(B, -1)
    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1) if "c0" in problem else 0.0

    # --------------------------------------------------------
    # 2. FSNET FEASIBILITY SEEKING (Differentiable Inner Loop)
    # --------------------------------------------------------
    # We initialize the seeking variables. We DO NOT detach them.
    v = v_0
    pg = pg_0
    qg = qg_0
    
    for _ in range(seek_steps):
        # A. Evaluate Constraints on Current State
        vp = quad_batch_stack(v, M_p); vq = quad_batch_stack(v, M_q)
        pf = quad_batch_stack(v, M_pf); qf = quad_batch_stack(v, M_qf)
        pt = quad_batch_stack(v, M_pt); qt = quad_batch_stack(v, M_qt)
        vc = quad_batch_stack(v, M_c); vs = quad_batch_stack(v, M_s)
        vv = quad_batch_stack(v, M_v)
        
        h_p = (pg @ C_g.T) - Pd_batch - vp
        h_q = (qg @ C_g.T) - Qd_batch - vq
        g_sf = (pf**2 + qf**2) - smax**2
        g_st = (pt**2 + qt**2) - smax**2
        g_ang_min = torch.tan(angmin) * vc - vs
        g_ang_max = vs - torch.tan(angmax) * vc
        g_v_max = vv - (Vmax**2)
        g_v_min = (Vmin**2) - vv
        g_pg_max = pg - pmax; g_pg_min = pmin - pg
        g_qg_max = qg - qmax; g_qg_min = qmin - qg
        
        # Sum up all violations to create the Feasibility Objective P(y)
        viol_loss = (
            h_p.pow(2).mean() + h_q.pow(2).mean() +
            F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean() +
            F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean() +
            F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean() +
            F.relu(g_pg_max).pow(2).mean() + F.relu(g_pg_min).pow(2).mean() +
            F.relu(g_qg_max).pow(2).mean() + F.relu(g_qg_min).pow(2).mean()
        )
        
        # B. Compute Differentiable Gradients
        # create_graph=True is the defining feature of FSNet! 
        # It allows the main optimizer to backpropagate THROUGH this solver loop.
        grad_v, grad_pg, grad_qg = torch.autograd.grad(
            viol_loss, (v, pg, qg), create_graph=True, retain_graph=True
        )
        
        # C. Take a gradient descent step (moving closer to feasibility)
        v = v - seek_lr * grad_v
        pg = pg - seek_lr * grad_pg
        qg = qg - seek_lr * grad_qg

    # --------------------------------------------------------
    # 3. FINAL TASK LOSS EVALUATION ON \hat{y} (Post-Seeking)
    # --------------------------------------------------------
    # The neural network is graded on how good the final point \hat{y} is.
    vp_f = quad_batch_stack(v, M_p); vq_f = quad_batch_stack(v, M_q)
    pf_f = quad_batch_stack(v, M_pf); qf_f = quad_batch_stack(v, M_qf)
    pt_f = quad_batch_stack(v, M_pt); qt_f = quad_batch_stack(v, M_qt)
    vc_f = quad_batch_stack(v, M_c); vs_f = quad_batch_stack(v, M_s)
    vv_f = quad_batch_stack(v, M_v)

    h_p_f = (pg @ C_g.T) - Pd_batch - vp_f
    h_q_f = (qg @ C_g.T) - Qd_batch - vq_f
    g_sf_f = (pf_f**2 + qf_f**2) - smax**2
    g_st_f = (pt_f**2 + qt_f**2) - smax**2
    g_ang_min_f = torch.tan(angmin) * vc_f - vs_f
    g_ang_max_f = vs_f - torch.tan(angmax) * vc_f
    g_v_max_f = vv_f - (Vmax**2)
    g_v_min_f = (Vmin**2) - vv_f
    g_pg_max_f = pg - pmax; g_pg_min_f = pmin - pg
    g_qg_max_f = qg - qmax; g_qg_min_f = qmin - qg

    # Objective Cost at the feasible point
    cost_per_gen = c2 * (pg ** 2) + c1 * pg + c0
    obj = cost_per_gen.sum(dim=1).mean()

    # Remaining Physical Violations at the feasible point
    final_primal_loss = (
        h_p_f.pow(2).mean() + h_q_f.pow(2).mean() +
        F.relu(g_sf_f).pow(2).mean() + F.relu(g_st_f).pow(2).mean() +
        F.relu(g_ang_min_f).pow(2).mean() + F.relu(g_ang_max_f).pow(2).mean() +
        F.relu(g_v_max_f).pow(2).mean() + F.relu(g_v_min_f).pow(2).mean() +
        F.relu(g_pg_max_f).pow(2).mean() + F.relu(g_pg_min_f).pow(2).mean() +
        F.relu(g_qg_max_f).pow(2).mean() + F.relu(g_qg_min_f).pow(2).mean()
    )

    total_loss = (weights["primal"] * final_primal_loss) + (weights["obj"] * obj)

    # --------------------------------------------------------
    # DIAGNOSTICS FOR BENCHMARKING
    # --------------------------------------------------------
    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_primal": final_primal_loss.detach().item(),
        "obj_cost": obj.detach().item(),
        
        "max_h_p": h_p_f.abs().max().detach().item(),
        "max_h_q": h_q_f.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf_f).max(), F.relu(g_st_f).max()).detach().item(),
        "max_v_viol": torch.max(F.relu(g_v_max_f).max(), F.relu(g_v_min_f).max()).detach().item(),
        "max_gen_viol": torch.max(
            torch.max(F.relu(g_pg_max_f).max(), F.relu(g_pg_min_f).max()),
            torch.max(F.relu(g_qg_max_f).max(), F.relu(g_qg_min_f).max())
        ).detach().item()
    }

    return total_loss, diagnostics

In [40]:
# You can use your identical baseline model architecture!
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model_fsnet = baselineQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

optimizer_fsnet = optim.Adam(model_fsnet.parameters(), lr=1e-3)

# Note: Because the network is trained on the POST-seeking point,
# we can usually weigh the objective much higher than standard PINNs!
loss_weights_fsnet = {
    "primal": 10.0,       
    "obj": 1.0,           
}

print("Starting Training: FSNet QCQP (Differentiable Seeking)")

for epoch in range(1000):
    Pd_batch = torch.clamp(gaussian_batch(problem["Pd"], batch_size=32), min=0.0)
    Qd_batch = torch.clamp(gaussian_batch(problem["Qd"], batch_size=32), min=0.0)
    
    optimizer_fsnet.zero_grad()
    
    # Run FSNet with 5 unrolled seeking steps
    # Note: If you get "Out Of Memory" (OOM) errors, lower seek_steps to 2 or 3.
    loss, diag = compute_fsnet_qcqp_smax_loss(
        model=model_fsnet, 
        Pd_batch=Pd_batch, 
        Qd_batch=Qd_batch, 
        problem=problem, 
        weights=loss_weights_fsnet,
        seek_steps=5,     
        seek_lr=1e-2      
    )
    
    loss.backward()
    
    # Clipping is mandatory because second-order gradients can explode
    torch.nn.utils.clip_grad_norm_(model_fsnet.parameters(), 10.0)
    optimizer_fsnet.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f}")

Starting Training: FSNet QCQP (Differentiable Seeking)
Epoch    0 | Cost: 2020.76 | Max P-Miss: 1.6887 | Max Q-Miss: 0.2695 | Max Gen Viol: 0.0002 | Max Thermal: 0.0000
Epoch  100 | Cost:    0.29 | Max P-Miss: 1.3340 | Max Q-Miss: 1.1258 | Max Gen Viol: 0.0002 | Max Thermal: 0.0000
Epoch  200 | Cost:   -0.00 | Max P-Miss: 1.3108 | Max Q-Miss: 0.6579 | Max Gen Viol: 0.0002 | Max Thermal: 1.1939
Epoch  300 | Cost:   -0.15 | Max P-Miss: 1.2116 | Max Q-Miss: 0.1872 | Max Gen Viol: 0.0002 | Max Thermal: 0.0000
Epoch  400 | Cost:    0.78 | Max P-Miss: 1.4011 | Max Q-Miss: 0.1944 | Max Gen Viol: 0.0002 | Max Thermal: 0.0000
Epoch  500 | Cost:    0.40 | Max P-Miss: 0.5668 | Max Q-Miss: 0.2848 | Max Gen Viol: 0.0001 | Max Thermal: 0.0000
Epoch  600 | Cost:   -0.09 | Max P-Miss: 0.6550 | Max Q-Miss: 0.2868 | Max Gen Viol: 0.0001 | Max Thermal: 0.0000
Epoch  700 | Cost:    0.00 | Max P-Miss: 0.7113 | Max Q-Miss: 0.0791 | Max Gen Viol: 0.0001 | Max Thermal: 0.0000
Epoch  800 | Cost:   -0.09 | Max 